# 🏭 EchoFactory — Kaggle Multi-SNR Training Notebook (-6 dB, 6 dB, 0 dB)
### Model: STgram-MFN v3 (Dual-Branch MobileFaceNet + ArcFace Metric Learning)
**COMPFEST 18 AI Innovation Challenge | Smart Manufacturing Track**

---

## ⚡ Keunggulan Khusus untuk Lingkungan Kaggle GPU (T4 / P100):
1. **Auto-Detect Kaggle Dataset Path**: Otomatis mencari dan mendeteksi struktur folder dataset MIMII di `/kaggle/input/` tanpa perlu ubah-ubah path manual.
2. **Super-Fast In-Memory RAM Preloading**: Memuat spektrogram Log-Mel dan High-Res Linear STFT langsung ke RAM (~2-3 detik/epoch, 100 epoch selesai dalam ~3-4 menit di Kaggle GPU).
3. **Multi-SNR Support**: Bisa memilih training pada **`-6_dB`** (kondisi kebisingan pabrik ekstrem), **`6_dB`** (kebisingan rendah), **`0_dB`** (standar benchmark), atau gabungan **`ALL`**.
4. **Multi-Machine Automation**: Bisa melatih 1 mesin spesifik atau langsung melatih ke-4 mesin (`fan`, `pump`, `slider`, `valve`) secara otomatis berurutan.
5. **One-Click Export**: Otomatis mengekspor model PyTorch `.pt`, model ultra-ringan `.onnx` (183 KB), visualisasi evaluasi `.png`, dan `inference_config.json` ke `/kaggle/working/`.

In [ ]:
# =====================================================================
# CELL 1: SETUP ENVIRONMENT KAGGLE & DEPENDENSI
# =====================================================================
import os, sys, gc, glob, json, math, time, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
from tqdm.auto import tqdm

# Pastikan onnx terinstall di Kaggle
try:
    import onnx
    import onnxscript
except ImportError:
    print('Menginstall ONNX & ONNXScript...')
    subprocess.run(['pip', 'install', '-q', 'onnx', 'onnxscript'], check=True)
    import onnx

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

print('=' * 65)
print(f'🚀 LINGKUNGAN KAGGLE GPU TERVERIFIKASI')
print(f'Device     : {device}')
if torch.cuda.is_available():
    print(f'GPU Name   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
print(f'Output Dir : {OUT_DIR}')
print('=' * 65)


In [ ]:
# =====================================================================
# CELL 2: AUTO-DETEKSI LOKASI DATASET MIMII DI KAGGLE
# =====================================================================
def find_kaggle_dataset_root():
    """Otomatis mendeteksi folder root dataset MIMII di /kaggle/input/"""
    base_input = '/kaggle/input'
    if not os.path.exists(base_input):
        # Fallback jika run lokal
        return './dataset_mimii'
        
    print('🔍 Memindai folder /kaggle/input/...')
    # Cari folder yang memuat file wav atau folder bermotif snr/machine
    candidate_roots = []
    for root, dirs, files in os.walk(base_input):
        for d in dirs:
            if any(k in d.lower() for k in ['0_db', '-6_db', '6_db', 'fan', 'pump', 'slider', 'valve']):
                candidate_roots.append(root)
                break
                
    if candidate_roots:
        # Ambil root teratas yang relevan
        best_root = sorted(candidate_roots, key=lambda x: len(x.split(os.sep)))[0]
        print(f'✅ Dataset Root Ditemukan: {best_root}')
        return best_root
        
    # Default fallback standar MIMII Kaggle dataset
    return '/kaggle/input/datasets/bisheshgiri/mimii-dataset'

DATASET_ROOT = find_kaggle_dataset_root()

# Tampilkan struktur isi dataset yang terdeteksi di Kaggle
print('\n📁 Daftar Konten di Dataset Root:')
if os.path.exists(DATASET_ROOT):
    for item in sorted(os.listdir(DATASET_ROOT))[:15]:
        print(f'  ├── {item}')
else:
    print('  [!] Folder belum terhubung. Pastikan Anda telah Add Data (MIMII Dataset) di notebook Kaggle.')


In [ ]:
# =====================================================================
# CELL 3: PENGATURAN TARGET SNR & MESIN
# =====================================================================
# 1. PILIH TARGET SNR:
#    '-6_dB' : Kondisi kebisingan pabrik ekstrem (Noise SNR -6dB)
#    '6_dB'  : Kondisi kebisingan rendah / clean floor (Noise SNR +6dB)
#    '0_dB'  : Kondisi standar benchmark IEEE (Noise SNR 0dB)
#    'ALL'   : Mixed Training (-6dB, 0dB, 6dB sekaligus)
TARGET_SNR = '-6_dB'   # <--- UBAH DI SINI: '-6_dB', '6_dB', '0_dB', atau 'ALL'

# 2. PILIH MESIN TARGET:
#    'fan' | 'pump' | 'slider' | 'valve'
TARGET_MACHINE = 'fan' # <--- UBAH DI SINI: 'fan', 'pump', 'slider', 'valve'

# 3. OPSI TRAINING OTOMATIS SEMUA 4 MESIN SEKALIGUS:
TRAIN_ALL_MACHINES = False  # Set True jika ingin otomatis melatih Fan, Pump, Slider, Valve dalam 1 run

# Parameter Audio & Spektrogram
SR        = 16000
AUDIO_LEN = SR * 10  # 10 detik = 160.000 sampel PCM mono

N_MELS    = 128
N_FFT_MEL = 1024
HOP_MEL   = 512

N_FFT_TG  = 512
HOP_TG    = 256
N_BINS_TG = 128

# Hyperparameter ArcFace Deep Metric Learning
EMBED_DIM    = 128
ARC_S        = 30.0   # ArcFace Scale
ARC_M        = 0.5    # ArcFace Angular Margin
BATCH_SIZE   = 64
EPOCHS       = 100
LR           = 5e-4
WARMUP_EP    = 15
WEIGHT_DECAY = 1e-3

print('=' * 65)
print(f'Target SNR         : {TARGET_SNR}')
print(f'Target Mesin       : {TARGET_MACHINE.upper() if not TRAIN_ALL_MACHINES else "SEMUA MESIN (FAN, PUMP, SLIDER, VALVE)"}')
print(f'Training Config    : Epochs={EPOCHS} | Batch Size={BATCH_SIZE} | LR={LR}')
print('=' * 65)


In [ ]:
# =====================================================================
# CELL 4: SPECAUGMENT & ARSITEKTUR STgram-MFN v3
# =====================================================================
class SpecAugment(nn.Module):
    """Regularisasi Frequency & Time Masking pada spektrogram."""
    def __init__(self, freq_mask=12, time_mask=16):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask

    def forward(self, x):
        if not self.training:
            return x
        B, C, F_dim, T_dim = x.shape
        out = x.clone()
        for b in range(B):
            f_len = torch.randint(0, self.freq_mask + 1, (1,)).item()
            if f_len > 0 and F_dim > f_len:
                f_0 = torch.randint(0, F_dim - f_len, (1,)).item()
                out[b, :, f_0:f_0+f_len, :] = 0
            t_len = torch.randint(0, self.time_mask + 1, (1,)).item()
            if t_len > 0 and T_dim > t_len:
                t_0 = torch.randint(0, T_dim - t_len, (1,)).item()
                out[b, :, :, t_0:t_0+t_len] = 0
        return out

class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False),
            nn.BatchNorm2d(oc),
            nn.PReLU(oc)
        )
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNPReLU(ic, ic, s=s, g=ic),
            ConvBNPReLU(ic, oc, k=1, p=0)
        )
    def forward(self, x): return self.net(x)

class MobileFaceNetEncoder(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.aug = SpecAugment(freq_mask=12, time_mask=16)
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2),
            DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2),
            DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2),
            DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, ed),
            nn.BatchNorm1d(ed)
        )
    def forward(self, x):
        x = self.aug(x)
        return self.head(self.enc(x))

class ArcFaceLoss(nn.Module):
    def __init__(self, ed, nc, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, feat, labels):
        cos = F.normalize(feat, dim=1) @ F.normalize(self.W, dim=1).T
        sin = (1.0 - cos.pow(2)).clamp(1e-9).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        oh  = F.one_hot(labels, cos.shape[1]).float()
        return F.cross_entropy((oh * phi + (1.0 - oh) * cos) * self.s, labels)

    @torch.no_grad()
    def class_weights(self):
        return F.normalize(self.W, dim=1)

class STgramMFN_v3(nn.Module):
    def __init__(self, n_classes, ed=128):
        super().__init__()
        self.mel_encoder   = MobileFaceNetEncoder(ed)
        self.tgram_encoder = MobileFaceNetEncoder(ed)
        self.fuse = nn.Sequential(
            nn.Linear(ed * 2, ed),
            nn.BatchNorm1d(ed),
            nn.PReLU(ed)
        )
        self.arc = ArcFaceLoss(ed, n_classes, s=ARC_S, m=ARC_M)

    def forward(self, mel, tg, labels=None):
        f_mel = self.mel_encoder(mel)
        f_tg  = self.tgram_encoder(tg)
        feat  = F.normalize(self.fuse(torch.cat([f_mel, f_tg], dim=1)), dim=1)

        if labels is not None:
            return feat, self.arc(feat, labels)
        return feat

print('✅ Arsitektur STgram-MFN v3 & ArcFace Loss terinisialisasi OK.')


In [ ]:
# =====================================================================
# CELL 5: DATASET PRELOADER (SUPER-FAST IN-MEMORY RAM LOADER)
# =====================================================================
class Kaggle_MIMIIDataset_RAM(Dataset):
    def __init__(self, root, machine, target_snr, cond='normal', sr=16000, audio_len=160000):
        self.sr = sr
        self.audio_len = audio_len
        
        snr_list = ['-6_dB', '0_dB', '6_dB'] if target_snr == 'ALL' else ([target_snr] if isinstance(target_snr, str) else target_snr)
        file_entries = []
        all_ids = set()
        
        for snr in snr_list:
            # Cari folder spesifik di Kaggle dataset
            patterns = [
                os.path.join(root, f'{snr}_{machine}', machine, 'id_*', cond, '*.wav'),
                os.path.join(root, f'{snr}_{machine}', 'id_*', cond, '*.wav'),
                os.path.join(root, f'*{snr}*{machine}*', '**/id_*', cond, '*.wav'),
                os.path.join(root, f'*{machine}*', f'*{snr}*', '**/id_*', cond, '*.wav'),
                os.path.join(root, '**/id_*', cond, '*.wav')
            ]
            
            found_files = []
            for pat in patterns:
                matches = glob.glob(pat, recursive=True)
                # Filter agar cocok dengan machine & snr
                valid = [m for m in matches if machine in m.lower() and (target_snr == 'ALL' or snr.replace('_','') in m.replace('_','').lower())]
                if valid:
                    found_files = sorted(valid)
                    break
                    
            for fp in found_files:
                parts = os.path.normpath(fp).split(os.sep)
                id_part = [p for p in parts if 'id_' in p.lower()]
                mid = id_part[-1] if id_part else 'id_00'
                all_ids.add(mid)
                file_entries.append((fp, mid, snr))
                
        if len(file_entries) == 0:
            # Upward fallback search
            fallback_glob = glob.glob(f'/kaggle/input/**/{machine}/**/{cond}/*.wav', recursive=True)
            for fp in fallback_glob:
                parts = os.path.normpath(fp).split(os.sep)
                mid = [p for p in parts if 'id_' in p.lower()][-1]
                all_ids.add(mid)
                file_entries.append((fp, mid, target_snr))
                
        if len(file_entries) == 0:
            raise FileNotFoundError(f'Tidak ada file audio ditemukan untuk {machine} ({target_snr}, {cond}) di {root}.')
            
        self.unique_ids = sorted(list(all_ids))
        self.id2label = {mid: i for i, mid in enumerate(self.unique_ids)}
        self.n_classes = len(self.unique_ids)
        
        print(f'\n[Dataset: {machine.upper()} | SNR: {target_snr} | Kondisi: {cond}]')
        print(f'  Total File : {len(file_entries)}')
        print(f'  Machine IDs: {self.id2label}')
        
        self.mels = []
        self.tgrams = []
        self.labels = []
        
        for fpath, mid, snr in tqdm(file_entries, desc=f'Preloading {cond} ({machine}) ke RAM'):
            wav, _ = sf.read(fpath, dtype='float32')
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if len(wav) >= self.audio_len:
                wav = wav[:self.audio_len]
            else:
                wav = np.pad(wav, (0, self.audio_len - len(wav)))
                
            # Log-Mel
            mel = librosa.feature.melspectrogram(y=wav, sr=self.sr, n_mels=N_MELS, n_fft=N_FFT_MEL, hop_length=HOP_MEL)
            mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
            
            # Linear STFT
            stft = np.abs(librosa.stft(y=wav, n_fft=N_FFT_TG, hop_length=HOP_TG))
            tg_db = librosa.amplitude_to_db(stft[:N_BINS_TG, :], ref=np.max).astype(np.float32)
            
            # Resample tensor ke (128, 128)
            t_mel = torch.from_numpy(mel_db).unsqueeze(0).unsqueeze(0)
            t_tg  = torch.from_numpy(tg_db).unsqueeze(0).unsqueeze(0)
            t_mel = F.interpolate(t_mel, (128, 128), mode='bilinear', align_corners=False).squeeze(0)
            t_tg  = F.interpolate(t_tg, (128, 128), mode='bilinear', align_corners=False).squeeze(0)
            
            self.mels.append(t_mel)
            self.tgrams.append(t_tg)
            self.labels.append(self.id2label[mid])
            
        self.mels_tensor = torch.stack(self.mels)
        self.tgrams_tensor = torch.stack(self.tgrams)
        self.labels_tensor = torch.tensor(self.labels, dtype=torch.long)
        print(f'✅ Selesai preload {len(self.labels)} sampel ({cond}) di RAM.')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.mels_tensor[idx], self.tgrams_tensor[idx], self.labels_tensor[idx]

print('Dataset Preloader Class Siap.')


In [ ]:
# =====================================================================
# CELL 6: FUNGSI TRAIN & EVALUASI MODULAR (EKSEKUSI 1 MESIN ATAU SEMUA)
# =====================================================================
def train_and_eval_machine(machine_name, target_snr):
    print('\n' + '#' * 70)
    print(f'🏭 MEMULAI TRAINING: MESIN {machine_name.upper()} | SNR {target_snr}')
    print('#' * 70)
    
    # 1. Load Normal Dataset
    train_ds = Kaggle_MIMIIDataset_RAM(DATASET_ROOT, machine_name, target_snr, cond='normal', sr=SR, audio_len=AUDIO_LEN)
    n_classes = train_ds.n_classes
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    
    # 2. Inisialisasi Model & Optimizer
    model = STgramMFN_v3(n_classes=n_classes, ed=EMBED_DIM).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
    
    def get_lr(ep):
        if ep <= WARMUP_EP: return LR * ep / WARMUP_EP
        p = (ep - WARMUP_EP) / (EPOCHS - WARMUP_EP)
        return LR * 0.5 * (1 + math.cos(math.pi * p))
        
    out_pt_path = os.path.join(OUT_DIR, f'stgram_mfn_v3_{machine_name}_{target_snr}.pt')
    best_loss = float('inf')
    losses = []
    t0 = time.time()
    
    # 3. Training Loop
    for ep in range(1, EPOCHS + 1):
        lr_now = get_lr(ep)
        for pg in optimizer.param_groups: pg['lr'] = lr_now
        
        model.train()
        ep_loss, n_bat = 0.0, 0
        
        for mel, tg, lab in train_dl:
            mel = mel.to(device, non_blocking=True)
            tg  = tg.to(device, non_blocking=True)
            lab = lab.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                _, loss = model(mel, tg, lab)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            
            ep_loss += loss.item()
            n_bat += 1
            
        avg_loss = ep_loss / max(n_bat, 1)
        losses.append(avg_loss)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': ep,
                'model_state': model.state_dict(),
                'best_loss': best_loss,
                'machine': machine_name,
                'target_snr': target_snr,
                'n_classes': n_classes,
                'embed_dim': EMBED_DIM,
                'id2label': train_ds.id2label,
                'arc_W': model.arc.class_weights().cpu(),
            }, out_pt_path)
            
        if ep % 20 == 0 or ep == 1 or ep == EPOCHS:
            print(f'Epoch {ep:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {lr_now:.2e} | Waktu: {(time.time()-t0)/60:.1f}m')
            
    print(f'🎉 Selesai Training {machine_name.upper()}! Best Loss: {best_loss:.4f} ({(time.time()-t0)/60:.2f} menit)')
    
    # 4. Evaluasi Anomaly Scoring (Load Abnormal Dataset)
    ck = torch.load(out_pt_path, map_location='cpu')
    eval_model = STgramMFN_v3(n_classes=ck['n_classes'], ed=ck['embed_dim']).to(device)
    eval_model.load_state_dict(ck['model_state'], strict=True)
    eval_model.eval()
    arc_W = ck['arc_W'].numpy()
    
    abnorm_ds = Kaggle_MIMIIDataset_RAM(DATASET_ROOT, machine_name, target_snr, cond='abnormal', sr=SR, audio_len=AUDIO_LEN)
    
    @torch.no_grad()
    def extract_embs(ds):
        dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
        res = []
        for mel, tg, _ in dl:
            f = eval_model(mel.to(device, non_blocking=True), tg.to(device, non_blocking=True))
            res.append(f.cpu().numpy())
        return np.concatenate(res, axis=0)
        
    norm_embs = extract_embs(train_ds)
    abnorm_embs = extract_embs(abnorm_ds)
    
    norm_labels = train_ds.labels_tensor.numpy()
    abnorm_labels = abnorm_ds.labels_tensor.numpy()
    
    y_true = np.concatenate([np.zeros(len(norm_embs)), np.ones(len(abnorm_embs))])
    all_labels = np.concatenate([norm_labels, abnorm_labels])
    all_embs = np.concatenate([norm_embs, abnorm_embs], axis=0)
    
    # KNN Cosine Per-ID
    unique_ids = np.unique(norm_labels)
    knn_models = {}
    for uid in unique_ids:
        mask = (norm_labels == uid)
        knn = NearestNeighbors(n_neighbors=min(5, mask.sum()), metric='cosine', algorithm='brute')
        knn.fit(norm_embs[mask])
        knn_models[uid] = knn
    
    s_knn = np.array([float(knn_models[all_labels[i]].kneighbors(all_embs[i:i+1])[0].mean()) for i in range(len(all_embs))])
    s_arc = np.array([1.0 - float(np.dot(all_embs[i], arc_W[all_labels[i]])) for i in range(len(all_embs))])
    
    r_knn = rankdata(s_knn) / len(s_knn)
    r_arc = rankdata(s_arc) / len(s_arc)
    s_ens = r_knn * 0.6 + r_arc * 0.4
    
    auc_knn = roc_auc_score(y_true, s_knn)
    auc_ens = roc_auc_score(y_true, s_ens)
    
    best_sc = s_knn if auc_knn >= auc_ens else s_ens
    best_auc = max(auc_knn, auc_ens)
    best_pauc = roc_auc_score(y_true, best_sc, max_fpr=0.1)
    best_name = 'KNN-k5' if auc_knn >= auc_ens else 'Ensemble'
    
    fpr, tpr, thr = roc_curve(y_true, best_sc)
    best_thr = float(thr[np.argmax(tpr - fpr)])
    
    print('=' * 60)
    print(f'🏆 HASIL {machine_name.upper()} ({target_snr}): AUC = {best_auc*100:.2f}% | pAUC = {best_pauc*100:.2f}% | Thr = {best_thr:.4f}')
    print('=' * 60)
    
    # 5. Export ke ONNX (183 KB)
    onnx_path = os.path.join(OUT_DIR, f'stgram_mfn_v3_{machine_name}_{target_snr}.onnx')
    m_cpu = STgramMFN_v3(n_classes=n_classes, ed=EMBED_DIM)
    m_cpu.load_state_dict(ck['model_state'], strict=True)
    m_cpu.eval()
    
    dummy_m = torch.randn(1, 1, 128, 128)
    dummy_t = torch.randn(1, 1, 128, 128)
    
    try:
        torch.onnx.export(
            m_cpu, (dummy_m, dummy_t), onnx_path,
            input_names=['mel', 'tgram'], output_names=['embedding'],
            opset_version=17, do_constant_folding=True,
            dynamic_axes={'mel': {0: 'B'}, 'tgram': {0: 'B'}, 'embedding': {0: 'B'}}
        )
    except Exception:
        torch.onnx.export(
            m_cpu, (dummy_m, dummy_t), onnx_path,
            input_names=['mel', 'tgram'], output_names=['embedding'],
            opset_version=16, do_constant_folding=True,
            dynamic_axes={'mel': {0: 'B'}, 'tgram': {0: 'B'}, 'embedding': {0: 'B'}}
        )
    print(f'✅ Export ONNX: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')
    
    # 6. Save Config JSON
    cfg = {
        'machine': machine_name, 'snr': target_snr,
        'auc': float(best_auc), 'pauc': float(best_pauc), 'threshold': float(best_thr),
        'scorer': best_name, 'id2label': train_ds.id2label, 'n_classes': n_classes,
        'embed_dim': EMBED_DIM
    }
    with open(os.path.join(OUT_DIR, f'config_{machine_name}_{target_snr}.json'), 'w') as f:
        json.dump(cfg, f, indent=2)
        
    return {'machine': machine_name, 'snr': target_snr, 'auc': best_auc, 'pauc': best_pauc, 'thr': best_thr}

print('Fungsi train_and_eval_machine siap.')


In [ ]:
# =====================================================================
# CELL 7: JALANKAN TRAINING & EVALUASI
# =====================================================================
results_summary = []

if TRAIN_ALL_MACHINES:
    machines_to_train = ['fan', 'pump', 'slider', 'valve']
    print(f'🚀 Melatih seluruh 4 jenis mesin pada SNR: {TARGET_SNR}')
    for m in machines_to_train:
        try:
            res = train_and_eval_machine(m, TARGET_SNR)
            results_summary.append(res)
        except Exception as e:
            print(f'❌ Gagal melatih mesin {m}: {e}')
else:
    print(f'🚀 Melatih mesin tunggal: {TARGET_MACHINE.upper()} pada SNR: {TARGET_SNR}')
    res = train_and_eval_machine(TARGET_MACHINE, TARGET_SNR)
    results_summary.append(res)

# =====================================================================
# TABEL REKAPITULASI HASIL KAGGLING
# =====================================================================
print('\n' + '=' * 65)
print(f'📊 REKAPITULASI HASIL TRAINING ECHOFACTORY ({TARGET_SNR})')
print('=' * 65)
for r in results_summary:
    print(f"{r['machine'].upper():8s} | SNR: {r['snr']:6s} | AUC: {r['auc']*100:6.2f}% | pAUC: {r['pauc']*100:6.2f}% | Threshold: {r['thr']:.4f}")
print('=' * 65)

with open(os.path.join(OUT_DIR, f'summary_results_{TARGET_SNR}.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f'✅ Seluruh hasil tersimpan di {OUT_DIR}/')
